In [ ]:
import os
import json
import pandas as pd
from pprint import pprint
import re

# Prompt Maker

This section explains how to use the PromptSyntDataMaker class with prompt templates and parameters. The application focuses on the Norwegian NACE classification and demonstrates how you can create your own customized application.

## PromptMaker
This abstract class is responsible for constructing and populating prompt templates.

In [ ]:
from statcodgen.prompt_maker import PromptMaker
# help(PromptMaker)

In [ ]:
prompt_template = 'Hi. [intro], [question]' # The class will examine patterns such as '-' or '[-]', as in the example.
values = {
    "intro": "Who are you?",
    "question": "Do you know where the nearest supermarket is?"
}



In [ ]:
prompt_basic = PromptMaker(
        prompt_template=prompt_template,
        plh_replace_map_dict=values,
        gap_ph_pattern='[-]' # Patterns must be specified using the open delimiter - close delimiter format.
)

In [ ]:
#help(prompt_basic)
prompt_basic.fill_prompt() # This main method fills the placeholders using the provided dictionary.

In [ ]:
prompt_template = 'Hi, *intro*, *question*'
values = {
    'question': 'Who are you?',
    'intro': "Do you know where the nearest supermarket is?"
}

prompt_basic = PromptMaker(
    prompt_template=prompt_template,
    plh_replace_map_dict=values, # new pattern
    gap_ph_pattern='*-*'
)

In [ ]:
prompt_basic.fill_prompt()

In [ ]:
# All keys specified in the prompt template must be provided (in this case, question and introduction).
prompt_basic.plh_replace_map_dict = {
        'question': 'Who are you?',
}

print(prompt_basic.fill_prompt())

In [ ]:
# But you can provide extra keys
prompt_basic.plh_replace_map_dict = {
    'question': 'Who are you?',
    'intro': "Do you know where the nearest supermarket is?",
    'extra': 'extra'
}
print(prompt_basic.fill_prompt())

## PromptSyntDataMaker

In [ ]:
from statcodgen.prompt_synt_data_maker import PromptSyntDataMaker
# help(PromptSyntDataMaker)

### Our strategy
This class implements the generation of synthetic data to train an AI model that classifies Norwegian Bokmål (NB) text according to the NACE standard. The prompt-making strategy uses nested templates and parameter dictionaries to generate synthetic data based on the explanatory notes of the standard.

To run the code above, your project root directory must have the following structure:
```
 root (dir):
    input (dir):
        *repository's all templates in synt_data_maker/input*
```
Or approach divide the prompt in three parts:
- **body**: where the role (body_intro), task (task), and output parameters (body_param) are defined.
    - input/body_template.json: basic template for writing the body.
    - input/body_param_template.json: basic template of the body parameters.
- **example**: where an example of the task is added.
    - input/example_template.json: basic template for writing the example.
- **objetive**: where the explanatory note is added.
    - input/objetive_template.json: basic template for writing the objective.

In [ ]:
root = os.path.abspath('') # Add the root directory where the `input` folder is located.
# print(root)
prompt_synt = PromptSyntDataMaker(
    root = root
)

In [ ]:
#  The full `prompt_template` is structured as follows:
prompt_template = prompt_synt.join_templates('[body]\n[example]\n[objetive]')
print('\nPrompt template\n',prompt_template)

In [ ]:
# PromptSyntDataMaker.write_prompt() parameters
print(prompt_basic.get_gap_keys(prompt_template, '**-**'))

There are nine placeholders to fill in the template. Five must be provided by the user, and four are parameterized.

1) **body_intro** (free text): Here, you must define the role of the model.

2) **task** (parameterized):
    - *generate*: Instructs the model to generate new sentences that fit a given description.
    - *process*: Instructs the model to extract all examples of economic activities included in a description.
    - *combine*: Instructs the model to generate new sentences that fit the description using specific keywords.
3) **language** (parameterized): Indicates the language in which the synthetic output data must be written.
    - *sp*: Generate sentences in Spanish.
    - *en*: Generate sentences in English.
    - *nb*: Generate sentences in Norwegian Bokmål.
4) **number** (free text): Number of examples the model must generate.
5) **len_constraint** (parameterized): Sentence length constraint.
    - *short*: Between 1–5 words.
    - *medium*: Between 5–15 words.
    - *long*: More than 15 words.
6) **output_format** (parameterized): Output format.
    - *lines*: Sentences must be separated by line breaks.
7) **example** (free text): An example of the expected output.
8) **description** (free text): Explanatory note.
9) **key_words** (free text, list format): A list of keywords. Do not include this parameter unless the task is *combine*.

In [ ]:
example = '''
Description:
Rice cultivation.

Key words:
['plantation', 'brown', 'rice', 'basmati']

Output:
Rice plantation.
Rice cultivation.
Brown rice plantation.
Basmati rice plantation.
Brown rice cultivation.
'''


In [ ]:
print(
    prompt_synt.write_prompt(
        body_intro = 'You are an expert in NACE classification',
        task = 'combine',
        language = 'sp',
        len_constraint = 'short',
        number = '5',
        output_format = 'lines',
        example = example,
        description = 'banana cultivation',
        key_words = '[banana, cultivation, plantation]'
    )
)

In [ ]:
# You can remove any parameter by setting it to `None`, which will remove the corresponding sentences from the prompt.
print(
    prompt_synt.write_prompt(
        body_intro = None,
        task = 'process',
        language = None,
        len_constraint = None,
        number = None,
        output_format = 'lines',
        example = None,
        description = 'banana cultivation',
        key_words = None
    )
)

### Custom your strategy

#### Make all gaps free text

In [ ]:
prompt_synt = PromptSyntDataMaker(
    root = root,
    name_param_dict=None # Not load the param dict
)

In [ ]:
print(
    prompt_synt.write_prompt(
        body_intro = 'You are an expert in ISCO classification',
        task = 'Give five jobs examples related to an economic activity',
        language = 'german',
        len_constraint = 'between 3 and 10',
        number = None,
        output_format = 'json',
        example = None,
        description = 'banana raise',
        key_words = None
    )
)

#### Make your own approach
All {name}_template.json documents in the input directory share the same structure:
``` python
{
"en": "This is an example [gap_to_fill_with_another_template], **gap_to_fill_with_str**",
"sp": "Esto es un ejemplo [gap_to_fill_with_another_template], **gap_to_fill_with_str**",
"{lang}": "{prompt_template}"
}
```
For example, PromptSyntDataMaker requires the following inputs:
```python
prompt_synt = PromptSyntDataMaker(
    root=None,
    name_root_template='prompt',
    name_param_dict='input/param_dict.json',
    prompt_lang='en',  # Indicates the language of the prompt
    template_pattern='[-]',  # Pattern used to identify nested templates
    gap_ph_pattern='**-**'  # Pattern used to identify text placeholders
)
```
In this case, the class retrieves the 'en' key from prompt_template.json, which must be located in the root/input directory. It then starts an iterative process to identify all text enclosed in [] and loads the corresponding 'en' templates from JSON files named {text}_template.json.

Let's look at an example. Suppose we want to automate a prompt-writing strategy to ask a model to write a poem. The prompt will have two parts: an introduction and a body, where the poem parameters are defined. The prompt can be written in both Spanish and English.

In [ ]:
# Step 1, we create the root directory with the required folder structure.
os.makedirs('example_new_approach/input', exist_ok=True)
new_root = os.path.abspath('example_new_approach')
print(os.path.exists(new_root))

In [ ]:
# Step 2: Create the base `poem_prompt_template.json` file.

structure = {
    "en": "**intro**\n[poem_body]",
    "sp": "**intro**\n[poem_body]",
}

poem_body = {
    "en": "The poem must be a **type**.\nMust be about **topic**.\nWrite only the poem, without any introduction.",
    "sp": "El poema debe ser **type**.\nDebe tratar sobre **topic**.\nEscribe solo el poema, sin ninguna introducción.",
}

with open(
    os.path.join(new_root, 'input', 'structure_template.json'),
    'w',
    encoding='utf-8'
) as f:  # REMEMBER: all template files must end with `_template.json`
    json.dump(structure, f, ensure_ascii=False, indent=4)

with open(
    os.path.join(new_root, 'input', 'poem_body_template.json'),
    'w',
    encoding='utf-8'
) as f:  # REMEMBER: the filename must match the gap name inside `[]` in `structure_template.json`
    json.dump(poem_body, f, ensure_ascii=False, indent=4)


In [ ]:
# Now everything is ready to generate the instance
poem_prompt_maker = PromptSyntDataMaker(
    root=new_root,
    name_root_template='structure',  # The name of the root prompt template
    name_param_dict=None,  # No parameter dictionary is used
    prompt_lang='en',  # Indicates the language of the prompt
    template_pattern='[-]',  # Pattern used to identify nested templates
    gap_ph_pattern='**-**'  # Pattern used to identify text placeholders
)


In [ ]:
# Fill the gaps using the parameter names defined in `**-**`
print(
    poem_prompt_maker.write_prompt(
        intro='You must write a poem.',
        type='romance',
        topic='love'
    )
)

In [ ]:
# If a parameter is not provided (set to None), the corresponding sentence will be removed
print(
    poem_prompt_maker.write_prompt(
        intro='You must write a poem.',
        type=None,
        topic=None
    )
)


In [ ]:
# Finally, you can also parameterize text by defining another .json file
poem_param_dict = {
    "en": {
        "topic": {
            "l": "love",
            "c": "city",
            "w": "war"
        },
    },
    "sp": {
        "topic": {
            "l": "amor",
            "c": "ciudad",
            "w": "guerra"
        },
    },
}

with open(os.path.join(new_root, 'input', 'poem_param.json'), 'w', encoding='utf-8') as f:
    json.dump(poem_param_dict, f, ensure_ascii=False, indent=4)

# Now use it to easily fill the text placeholders
poem_prompt_maker = PromptSyntDataMaker(
    root=new_root,
    name_root_template='structure',  # The name of the root prompt template
    name_param_dict='input/poem_param.json',  # Relative path to the parameter dictionary
    prompt_lang='sp',  # Indicates the language of the prompt
    template_pattern='[-]',  # Pattern used to identify nested templates
    gap_ph_pattern='**-**'  # Pattern used to identify text placeholders
)

print('Example with love\n')
print(
    poem_prompt_maker.write_prompt(
        intro='Tienes que escribir un poema.',
        type='romance',
        topic='l'
    )
)

print('\nExample with war\n')
print(
    poem_prompt_maker.write_prompt(
        intro='Tienes que escribir un poema.',
        type='romance',
        topic='w'
    )
)


# Process explicative notes from a standard
The aim of this class is to prepare the explicative notes of a standard for generating a dict with the notes, examples and key word for each code. First it will show how we have worked with the Norwegian NACE standard.

## ProcessNACENB

In this tutorial, three documents are used:

- **klass-version-3218-codes.csv**: The Norwegian Standard Industrial Classification (NACEnb). You can download it [here](https://www.ssb.no/klass/klassifikasjoner/6).

- **train_norwaydata.csv**: A NACEnb-labeled training dataset created by [Statistics Norway (Statistisk sentralbyrå)](https://www.ssb.no/en). It was shared within Standard Working Group 5.

- **NACE Rev. 2.1 - Index entries - Overview_translated_to_NB_with_deepl_complete.csv**: The NACE index published by Eurostat and translated into Norwegian Bokmål using [DeepL](https://www.deepl.com/es/translator). Please contact us if you would like access to this file.


In [ ]:
from statcodgen.process_notes import ProcessNACENB
# help(ProcessNACENB)

In [ ]:
# Load docs
doc_path = r""  #path with the docs
nb_notes = 'klass-version-3218-codes.csv'
organic_train_set_nb = 'train_norwaydata.csv'
index_nb = 'NACE Rev. 2.1 - Index entries - Overview_translated_to_NB_with_deepl_complete.csv'

notes_df = pd.read_csv(
    os.path.join(doc_path, nb_notes),
    dtype='str',
    encoding='latin-1',
    sep=';',
)
train_df = pd.read_csv(
    os.path.join(doc_path, organic_train_set_nb),
    dtype='str'
)
index_df = pd.read_csv(
    os.path.join(doc_path, index_nb),
    dtype='str'
)


In [ ]:
# Now generate a sample dictionary to create synthetic data
process_nb = ProcessNACENB(
    notes_df=notes_df[notes_df['level'] == "4"],
    sample=3,  # Take only three classes
    random_state=23,  # For reproducibility
    examples_df=index_df,  # Initially use the index to extract keywords
    n_examples=10  # Number of real examples and keyword-based examples
)

process_nb.set_descriptions(
    col_c='code',
    col_d_l=['name', 'notes']
)

process_nb.set_key_words(
    col_c='CODE',
    col_k='KEYWORD_NB'
)

process_nb.example_df = train_df  # Now use the real labeled examples

process_nb.set_examples(
    col_c='nace_21_4digit_code',
    col_d='text_for_coding'
)

nace_nb_dict = process_nb.get_dict_data()

In [ ]:
pprint(nace_nb_dict)

### Customize Your Strategy

`NotesProcess` is designed so that advanced users only need to override one method:

`preprocess_text(text: str) -> str`

In [ ]:
from statcodgen.process_notes import NotesProcess

class MyStandardProcessor(NotesProcess):
    @staticmethod
    def preprocess_text(text: str) -> str:
        text = text.lower()
        text = re.sub(r"\s+", " ", text).strip()
        return text

In [ ]:
notes_df = pd.DataFrame({
    "code": ["A", "B"],
    "title": ["Agriculture", "Mining"],
    "notes": ["Explanatory note A...", "Explanatory note B..."]
})

examples_df = pd.DataFrame({
    "code": ["A", "A", "B"],
    "example": ["Crop farming", "Animal breeding", "Coal extraction"],
    "keywords": ["farm crops", "cattle breeding", "coal mine"]
})

processor = MyStandardProcessor(
    notes_df=notes_df,
    examples_df=examples_df,
    n_examples=10  # Maximum number of examples and keywords to extract from examples_df
)

notes_dict = processor.get_dict_data(
    col_notes_c="code",
    col_notes_d_l=["title", "notes"],
    col_eg_c="code",       # Column in examples_df containing the code
    col_eg_d="example",    # Column in examples_df containing the examples
    col_eg_k="keywords"    # Column in examples_df containing the keywords
)

In [ ]:
pprint(notes_dict)

## Generate Synthetic Data for Your Classification

This class encapsulates the full pipeline for generating synthetic data following our strategy. It is designed to receive a data dictionary (such as one created by `ProcessNACENB`) and iteratively query a model via the API using `PromptSyntDataMaker`.

It is optimized to work with our templates and can be adapted to any classification task.



In [ ]:
model = "gpt-oss:120b"
YOUR_API_KEY = ""#

In [ ]:
from statcodgen.synt_data_maker import OnyxiaSyntDataGenerator

In [ ]:
generator = OnyxiaSyntDataGenerator(
    root=root,  # Project root (must contain the ./input/ directory with all .json templates)
    api_key=YOUR_API_KEY,  # Onyxia API key
    model=model,  # Available Onyxia model name
    label_contents_map_dict=nace_nb_dict,  # Output from

    body_intro="You are an expert in statistical classification.",  # Free text
    task="generate",  # Task type: generate (create new data from notes), process (extract examples from text), combine (generate examples using notes and keywords)
    n_responses='10',  # Number of generated examples per label

    p_lang="en",  # Prompt template language (en or sp)
    o_lang="nb",  # Requested output language (en, sp, or nb)
    output_format="lines",  # Output format: examples separated by line breaks
    len_constraint="medium",  # Output length constraint (short, medium, or long)

    eg_strategy="dynamic",  # Example strategy: static, dynamic, or mixed
    eg_filename="static_examples.json"  # Filename for static examples (if used)
)

eg_strategy indicates to the class how example gap in prompt template must be filled.
- **dynamic**: Only include the examples in the label_contents_map_dict. If there aren't any, don't include anything.
- **static**: Load the same example for each task from static_examples.json
- **mixed**: Take an organic example if possible, otherwise take the example from the prompt template..

In [ ]:
df = generator.get_synt_data_df()

In [ ]:
df